In [2]:
import ollama

In [3]:
models = ollama.list()

for model in models["models"]:
    print(model["model"])

phi3:latest


In [4]:
gpu = input("Enter GPU name: ")

print("Searching for:", gpu)

Searching for: RTX 3060


In [5]:
from duckduckgo_search import DDGS

results = []

with DDGS() as ddgs:
    search_results = ddgs.text(
        f"{gpu} price Egypt",
        max_results=20
    )

    for result in search_results:
        results.append({
            "title": result["title"],
            "url": result["href"],
            "snippet": result["body"]
        })

results[:5]

C:\Users\Youssef Abdallah\AppData\Local\Temp\ipykernel_21648\4095210011.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[]

In [6]:
import pandas as pd

df = pd.DataFrame(results)

df

""


In [7]:
import json
import re
import ollama
import pandas as pd

data_text = df.to_json(orient="records", force_ascii=False)

prompt = f"""
You are a product price extraction assistant.

Target GPU: {gpu}

Search results:
{data_text}

Extract available GPU products and their prices in Egypt into a JSON list.
Output format:
[
  {{
    "store": "Store Name",
    "product": "GPU Full Name",
    "price": 15000,
    "currency": "EGP",
    "url": "https://..."
  }}
]

If no relevant prices are found in the search results, output: []
"""

response = ollama.chat(
    model="phi3:latest",
    format="json",  # <--- CRITICAL: forces strictly valid JSON
    messages=[{"role": "user", "content": prompt}]
)

raw = response["message"]["content"].strip()

# Safely extract JSON brackets [ ... ] or { ... }
match = re.search(r"(\[[\s\S]*\]|\{[\s\S]*\})", raw)

if match:
    try:
        data = json.loads(match.group(0))
        if isinstance(data, dict):
            # If wrapped in a dict like {"products": [...]}, get the inner list
            for val in data.values():
                if isinstance(val, list):
                    data = val
                    break
        price_df = pd.DataFrame(data if isinstance(data, list) else [data])
    except Exception as e:
        print("JSON parse error:", e)
        price_df = pd.DataFrame(columns=["store", "product", "price", "currency", "url"])
else:
    print("No JSON found in response.")
    price_df = pd.DataFrame(columns=["store", "product", "price", "currency", "url"])

output_file = f"{gpu.replace(' ', '_')}_prices.xlsx"

price_df.to_excel(
    output_file,
    index=False,
    sheet_name="GPU Prices"
)

print(f"Excel file created successfully: {output_file}")


Excel file created successfully: RTX_3060_prices.xlsx
